# TFT electricity runner for Colab

This notebook clones the repo, builds the dataset, runs a smoke test, then trains, predicts, evaluates, and compares models while saving outputs to Google Drive and locally in the repo.

In [ ]:
# Imports
import os
import shutil
import subprocess
from pathlib import Path
import torch


In [ ]:
# Configuration

REPO_URL = "https://github.com/HannaVallner/tft_electricity.git"
REPO_DIR = "tft_electricity"

# Colab paths
COLAB_WORKING = Path("/content")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "tft_electricity_seed_runs"
DRIVE_OUTPUT_COPY = DRIVE_PROJECT_ROOT / "outputs"
DRIVE_ZIP_PATH = DRIVE_PROJECT_ROOT / "tft_outputs_latest.zip"

USE_DRIVE = False

RUN_SMOKE_TEST = False
RUN_FULL_TRAINING = True

# Random seeds to run. For quick smoke tests, this can be shortened.
RANDOM_SEEDS = [0, 1, 2, 3, 4]

SMOKE_MODELS = ["baseline"]
FULL_MODELS = ["baseline", "no_lstm", "no_attention", "mlp_features", "transformer_only"]

SMOKE_TRAIN_IDS = 20
SMOKE_VALID_IDS = 20
SMOKE_EPOCHS = 2

FULL_EPOCHS = 100
PRINT_EVERY = 1000


In [ ]:
# Mount Google Drive and clone repo

if USE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
    DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_OUTPUT_COPY.mkdir(parents=True, exist_ok=True)

COLAB_WORKING.mkdir(parents=True, exist_ok=True)
os.chdir(COLAB_WORKING)

if REPO_DIR.exists():
    print(f"Repo already exists: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Current working directory:", Path.cwd())


In [ ]:
# Install requirements

requirements_path = REPO_DIR / "requirements.txt"
!pip install -r requirements.txt

In [ ]:
# Create output folders and Drive safety helpers

LOCAL_OUTPUT_ROOT = Path("outputs")
LOCAL_CHECKPOINTS = LOCAL_OUTPUT_ROOT / "checkpoints"
LOCAL_PREDICTIONS = LOCAL_OUTPUT_ROOT / "predictions"
LOCAL_METRICS = LOCAL_OUTPUT_ROOT / "metrics"
LOCAL_PLOTS = LOCAL_OUTPUT_ROOT / "plots"

for p in [LOCAL_CHECKPOINTS, LOCAL_PREDICTIONS, LOCAL_METRICS, LOCAL_PLOTS]:
    p.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    DRIVE_OUTPUT_COPY.mkdir(parents=True, exist_ok=True)


def copy_to_drive_outputs(local_path: Path):
    """Copy one artifact into Google Drive with the same path relative to outputs/."""
    if not USE_DRIVE:
        return

    local_path = Path(local_path)
    if not local_path.exists():
        return

    try:
        relative_path = local_path.relative_to(LOCAL_OUTPUT_ROOT)
        destination = DRIVE_OUTPUT_COPY / relative_path
    except ValueError:
        destination = DRIVE_OUTPUT_COPY / local_path.name

    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_path, destination)


def sync_outputs_to_drive():
    """Mirror local outputs/ into Google Drive."""
    if not USE_DRIVE:
        return

    DRIVE_OUTPUT_COPY.mkdir(parents=True, exist_ok=True)
    for local_file in LOCAL_OUTPUT_ROOT.rglob("*"):
        if local_file.is_file():
            copy_to_drive_outputs(local_file)


def zip_outputs(label=""):
    """Create/update a downloadable ZIP in Google Drive."""
    if not LOCAL_OUTPUT_ROOT.exists():
        print("No outputs folder yet; skipping ZIP.")
        return

    if USE_DRIVE:
        sync_outputs_to_drive()
        zip_path = DRIVE_ZIP_PATH
    else:
        zip_path = Path("/content/tft_outputs_latest.zip")

    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()

    archive_base = zip_path.with_suffix("")
    shutil.make_archive(str(archive_base), "zip", root_dir=LOCAL_OUTPUT_ROOT)

    label_text = f" after {label}" if label else ""
    print(f"Saved safety ZIP{label_text}: {zip_path}")


def save_artifacts_after_step(paths, label):
    """Copy important files to Drive and optionally refresh the ZIP."""
    if USE_DRIVE:
        for path in paths:
            copy_to_drive_outputs(Path(path))

In [ ]:
# Check GPU

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Build dataset if not yet present

data_path = Path("data/electricity_processed.csv")
if not data_path.exists():
    !python src/create_dataset.py
else:
    print(f"Dataset already exists: {data_path}")

In [ ]:
# Run a smoke test if enabled

if RUN_SMOKE_TEST:
    for seed in RANDOM_SEEDS:
        for model_name in SMOKE_MODELS:
            print(f"===== SMOKE TEST: {model_name} | seed {seed} =====")

            ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
            hist = Path(f"outputs/metrics/{model_name}_seed_{seed}_training_history.json")

            if ckpt.exists() and hist.exists():
                print(f"Skipping {model_name} seed {seed}: checkpoint and history already exist.")
            else:
                !PYTHONPATH=.:./src python src/train.py \
                    --model {model_name} \
                    --data_path data/electricity_processed.csv \
                    --save_dir outputs/checkpoints \
                    --metrics_dir outputs/metrics \
                    --num_epochs {SMOKE_EPOCHS} \
                    --num_train_ids {SMOKE_TRAIN_IDS} \
                    --num_valid_ids {SMOKE_VALID_IDS} \
                    --print_every {PRINT_EVERY} \
                    --seed {seed}

            save_artifacts_after_step(
                [ckpt, hist],
                label=f"smoke training {model_name} seed {seed}",
            )

In [ ]:
# Full training if enabled

if RUN_FULL_TRAINING:
    for seed in RANDOM_SEEDS:
        for model_name in FULL_MODELS:
            print(f"===== FULL TRAINING: {model_name} | seed {seed} =====")

            ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
            hist = Path(f"outputs/metrics/{model_name}_seed_{seed}_training_history.json")

            if ckpt.exists() and hist.exists():
                print(f"Skipping {model_name} seed {seed}: checkpoint and history already exist.")
            else:
                !PYTHONPATH=.:./src python src/train.py \
                    --model {model_name} \
                    --data_path data/electricity_processed.csv \
                    --save_dir outputs/checkpoints \
                    --metrics_dir outputs/metrics \
                    --num_epochs {FULL_EPOCHS} \
                    --print_every {PRINT_EVERY} \
                    --seed {seed}

            save_artifacts_after_step(
                [ckpt, hist],
                label=f"full training {model_name} seed {seed}",
            )

In [ ]:
# Predict for available checkpoints

models_to_predict = FULL_MODELS if RUN_FULL_TRAINING else SMOKE_MODELS

for seed in RANDOM_SEEDS:
    for model_name in models_to_predict:
        ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
        if not ckpt.exists():
            print(f"Skipping {model_name} seed {seed}: checkpoint not found.")
            continue

        output_path = Path(f"outputs/predictions/{model_name}_seed_{seed}_predictions.csv")

        if output_path.exists():
            print(f"Skipping prediction for {model_name} seed {seed}: predictions already exist.")
        else:
            print(f"===== PREDICT: {model_name} | seed {seed} =====")
            !PYTHONPATH=.:./src python src/predict.py \
                --model {model_name} \
                --data_path data/electricity_processed.csv \
                --checkpoint_path {ckpt} \
                --output_path {output_path} \
                --batch_size 64

        save_artifacts_after_step(
            [output_path],
            label=f"prediction {model_name} seed {seed}",
        )

In [ ]:
# Evaluate each model and seed

models_to_predict = FULL_MODELS
for seed in RANDOM_SEEDS:
    for model_name in models_to_predict:
        ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
        pred = Path(f"outputs/predictions/{model_name}_seed_{seed}_predictions.csv")
        metrics_path = Path(f"outputs/metrics/{model_name}_seed_{seed}_metrics.json")

        if not ckpt.exists() or not pred.exists():
            print(f"Skipping {model_name} seed {seed}: missing checkpoint or predictions.")
            continue


        print(f"===== EVALUATE: {model_name} | seed {seed} =====")
        !PYTHONPATH=.:./src python src/evaluate.py \
            --model {model_name} \
            --data_path data/electricity_processed.csv \
            --checkpoint_path {ckpt} \
            --predictions_path {pred} \
            --metrics_path {metrics_path} \
            --plots_dir outputs/plots \
            --batch_size 64 \
            --seed {seed}

        plot_paths = list(Path("outputs/plots").glob(f"{model_name}_seed_{seed}_*.pdf"))
        paths_to_save = [metrics_path] + plot_paths

        save_artifacts_after_step(
            paths_to_save,
            label=f"evaluation {model_name} seed {seed}",
        )


In [ ]:
# Compare models across seeds

!PYTHONPATH=.:./src python src/compare_models.py \
    --metrics_dir outputs/metrics \
    --plots_dir outputs/plots \
    --predictions_dir outputs/predictions \
    --data_path data/electricity_processed.csv

comparison_files = (
    list(Path("outputs/metrics").glob("model_comparison*.*"))
    + list(Path("outputs/metrics").glob("calibration_summary.*"))
)

comparison_plots = (
    list(Path("outputs/plots").glob("compare_*.pdf"))
    + list(Path("outputs/plots").glob("combined_reliability*.pdf"))
    + list(Path("outputs/plots").glob("representative_seed_*_p50_comparison.pdf"))
)

save_artifacts_after_step(
    comparison_files + comparison_plots,
    label="final model comparison",
)

In [ ]:
# Zip outputs for download

zip_outputs("final run")
if USE_DRIVE:
    print(f"Final Drive ZIP: {DRIVE_ZIP_PATH}")
    print(f"Drive output folder: {DRIVE_OUTPUT_COPY}")
else:
    print("Final local ZIP: /content/tft_outputs_latest.zip")